# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 19.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=f62eadd3-9149-4c60-8014-0ef662860065
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 78.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 6.68MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)

    if rand_init:
      delta = torch.empty_like(x).uniform_(-eps, eps) #LK: check range -- uniform, -[eps, eps]
    else:
      delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    for _ in range(steps):
      grad = input_grad(model, x, delta, y)
      delta = delta + alpha * grad.sign()
      delta = clip_box(x, delta.clamp(-eps, eps))

      #LK: do we have to stop early

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)

    if rand_init:

      delta = torch.empty_like(x).normal_()

      # in the unlikely chance that delta is [0, 0, ..0] .... putting a min
      # so we don't run into divide by 0 errors
      n = l2_norm(delta).clamp(min=1e-12)
      r = torch.rand_like(n)
      delta = delta * (r / n) * eps
    else:
      delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    for _ in range(steps):
      grad = input_grad(model, x, delta, y)
      delta = delta + alpha * grad / l2_norm(grad).clamp(min=1e-12)
      # projection
      # l2_norm.clamp(min=eps) is equal to max(l2norm, eps)
      delta = delta * eps / l2_norm(delta).clamp(min=eps)
      delta = clip_box(x, delta)

    return (x + delta).detach()

def fgsm(model, x, y, eps):
    return pgd_linf(model, x, y, eps, eps, 1, False)


In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10, eps=-1):
    # The attack for one norm with the default eps and step size.
    if eps == -1:
      eps = EPS[norm] # use default eps
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, eps, alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, inc_fgsm=False, eps=-1, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    if inc_fgsm:
      correct = {k: 0 for k in ("clean", *NORMS, "fgsm", "union")}
    else:
      correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps, eps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust

        if inc_fgsm:
          robust = clean.clone()
          x_adv = fgsm(model, x, y, eps)
          with torch.no_grad():
              robust &= model(x_adv).argmax(1) == y
          correct["fgsm"] += robust.sum().item()
          union &= robust

        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [9]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()

    # For a non-fine tuned model (this is for problem 1 part a)
    if norm == "clean":
      x_adv = x
    else:
      x_adv = attack(model, x, y, norm)

    model.train()

    return F.cross_entropy(model(x_adv), y)


# PROBLEM 1

In [13]:
model = adv_train(make_model(), single_norm_loss, epochs=5, norm="clean")
model_linf1 = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")

epoch 1: loss 1.883 (8 s)
epoch 2: loss 1.550 (13 s)
epoch 3: loss 1.381 (18 s)
epoch 4: loss 1.219 (23 s)
epoch 5: loss 1.127 (29 s)
epoch 1: loss 2.187 (43 s)
epoch 2: loss 2.104 (86 s)
epoch 3: loss 2.057 (128 s)
epoch 4: loss 2.036 (171 s)
epoch 5: loss 2.021 (214 s)


In [17]:
eps_list = [2/255.0, 4/255.0, 8/255.0]
for eps in eps_list:
  print(f"Testing clean model with eps of {eps:.5f}")
  test_model(model, inc_fgsm=True, eps=eps)
  print(f"Testing l_inf model with eps of {eps:.5f}")
  test_model(model_linf1, inc_fgsm=True, eps=eps)

Testing clean model with eps of 0.00784
clean 59.2  linf 20.9  l2 58.2  l1 59.2  fgsm 26.2  union 20.9   (133 s)
Testing l_inf model with eps of 0.00784
clean 36.6  linf 33.7  l2 36.4  l1 36.5  fgsm 33.7  union 33.7   (131 s)
Testing clean model with eps of 0.01569
clean 59.2  linf 3.7  l2 56.8  l1 59.2  fgsm 10.2  union 3.7   (131 s)
Testing l_inf model with eps of 0.01569
clean 36.6  linf 29.5  l2 36.4  l1 36.5  fgsm 29.5  union 29.1   (131 s)
Testing clean model with eps of 0.03137
clean 59.2  linf 0.1  l2 54.6  l1 58.9  fgsm 2.2  union 0.1   (131 s)
Testing l_inf model with eps of 0.03137
clean 36.6  linf 23.7  l2 36.1  l1 36.5  fgsm 24.3  union 23.6   (131 s)


# PROBLEM 3

In [24]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (44 s)
epoch 2: loss 1.181 (42 s)
epoch 3: loss 1.152 (43 s)
epoch 4: loss 1.126 (42 s)
epoch 5: loss 1.097 (43 s)
epoch 6: loss 1.068 (43 s)
epoch 7: loss 1.049 (43 s)
epoch 8: loss 1.030 (43 s)
epoch 9: loss 1.013 (43 s)
epoch 10: loss 1.008 (43 s)
clean 78.6  linf 45.1  l2 56.7  l1 20.1  union 20.0   (129 s)


{'clean': 78.6, 'linf': 45.1, 'l2': 56.7, 'l1': 20.1, 'union': 20.0}

In [25]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (43 s)
epoch 2: loss 0.826 (43 s)
epoch 3: loss 0.787 (43 s)
epoch 4: loss 0.754 (43 s)
epoch 5: loss 0.714 (43 s)
epoch 6: loss 0.674 (43 s)
epoch 7: loss 0.631 (43 s)
epoch 8: loss 0.610 (43 s)
epoch 9: loss 0.591 (43 s)
epoch 10: loss 0.583 (43 s)
clean 82.5  linf 32.2  l2 60.1  l1 27.7  union 24.4   (130 s)


{'clean': 82.5, 'linf': 32.2, 'l2': 60.1, 'l1': 27.7, 'union': 24.4}

In [26]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.231 (43 s)
epoch 2: loss 0.897 (43 s)
epoch 3: loss 0.578 (43 s)
epoch 4: loss 0.461 (43 s)
epoch 5: loss 0.338 (43 s)
epoch 6: loss 0.274 (43 s)
epoch 7: loss 0.217 (43 s)
epoch 8: loss 0.198 (43 s)
epoch 9: loss 0.173 (43 s)
epoch 10: loss 0.175 (43 s)
clean 84.6  linf 0.0  l2 0.0  l1 23.9  union 0.0   (130 s)


{'clean': 84.6, 'linf': 0.0, 'l2': 0.0, 'l1': 23.9, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [10]:
def ramp_loss(model, x, y, lam=0.5):
    # print("i'm here")
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()

    linf = attack(model, x, y, "linf")
    l1 = attack(model, x, y, "l1")

    model.train()

    linf = model(linf)
    l1 = model(l1)

    linf_loss = F.cross_entropy(linf, y, reduction="none")
    l1_loss = F.cross_entropy(l1, y, reduction="none")
    lce = torch.maximum(linf_loss, l1_loss).mean()

    # kl only for correctly classified images under linf
    correct = torch.where(linf.argmax(dim=1) == y)[0]

    if len(correct) == 0:
      return lce
    else:
      log_pq = F.log_softmax(linf[correct], dim=1)
      log_pr = F.log_softmax(l1[correct], dim=1)

      pq = log_pq.exp()

      lkl = (pq * (log_pq - log_pr)).sum(dim=1).mean()

      return lce + lam * lkl


def gradient_projection(g_n, g_a):
    # print("i am also here")
    # TODO: GP(g_n, g_a) for one parameter tensor
    dot = (g_n * g_a).sum()
    norm_gn = torch.norm(g_n)
    norm_ga = torch.norm(g_a)

    cos = dot / (norm_gn * norm_ga).clamp(min=1e-12) # prevent error
    if cos > 0:
      return cos * g_n
    else:
      return torch.zeros_like(g_n)



In [27]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (91 s)
epoch 2: loss 1.331 (91 s)
epoch 3: loss 1.304 (91 s)
epoch 4: loss 1.282 (91 s)
epoch 5: loss 1.254 (91 s)
epoch 6: loss 1.227 (92 s)
epoch 7: loss 1.199 (91 s)
epoch 8: loss 1.186 (91 s)
epoch 9: loss 1.172 (91 s)
epoch 10: loss 1.165 (91 s)
clean 78.9  linf 41.0  l2 62.0  l1 44.9  union 39.2   (130 s)


{'clean': 78.9, 'linf': 41.0, 'l2': 62.0, 'l1': 44.9, 'union': 39.2}

In [28]:
# from google.colab import runtime
# runtime.unassign()


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [11]:
def my_loss(model, x, y, lam=0.5, alpha=0.5):
    model.eval()

    linf = attack(model, x, y, "linf")
    l1 = attack(model, x, y, "l1")

    model.train()

    linf = model(linf)
    l1 = model(l1)

    linf_loss = F.cross_entropy(linf, y, reduction="none")
    l1_loss = F.cross_entropy(l1, y, reduction="none")

    # NEW THING!!!!!!!!
    with torch.no_grad():
      # where was linf attack successful
      linf_attack_success = (linf.argmax(dim=1) != y)

    max_loss = torch.maximum(linf_loss, l1_loss)

    per_image_loss = torch.where(
        linf_attack_success,
        alpha * linf_loss + (1 - alpha) * max_loss,
        max_loss
        )
    lce = per_image_loss.mean()

     # kl only for correctly classified images under linf
    correct = torch.where(linf.argmax(dim=1) == y)[0]

    if len(correct) == 0:
      return lce
    else:
      log_pq = F.log_softmax(linf[correct], dim=1)
      log_pr = F.log_softmax(l1[correct], dim=1)

      pq = log_pq.exp()

      lkl = (pq * (log_pq - log_pr)).sum(dim=1).mean()

      return lce + lam * lkl


In [12]:
model_mine1 = adv_train(load_base_model(), my_loss)
results_mine1 = test_model(model_mine1)

model_mine2 = adv_train(load_base_model(), my_loss, seed=1)
results_mine2 = test_model(model_mine2)

model_ramp2 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp2)


epoch 1: loss 1.399 (86 s)
epoch 2: loss 1.301 (175 s)
epoch 3: loss 1.272 (264 s)
epoch 4: loss 1.246 (352 s)
epoch 5: loss 1.217 (441 s)
epoch 6: loss 1.188 (529 s)
epoch 7: loss 1.159 (618 s)
epoch 8: loss 1.144 (707 s)
epoch 9: loss 1.130 (795 s)
epoch 10: loss 1.124 (883 s)
clean 76.3  linf 43.5  l2 61.5  l1 46.2  union 41.6   (135 s)
epoch 1: loss 1.398 (89 s)
epoch 2: loss 1.303 (178 s)
epoch 3: loss 1.268 (266 s)
epoch 4: loss 1.239 (354 s)
epoch 5: loss 1.211 (443 s)
epoch 6: loss 1.184 (531 s)
epoch 7: loss 1.161 (619 s)
epoch 8: loss 1.145 (708 s)
epoch 9: loss 1.134 (796 s)
epoch 10: loss 1.124 (884 s)
clean 77.0  linf 44.4  l2 61.9  l1 47.2  union 42.3   (135 s)
epoch 1: loss 1.434 (95 s)
epoch 2: loss 1.333 (191 s)
epoch 3: loss 1.299 (286 s)
epoch 4: loss 1.274 (380 s)
epoch 5: loss 1.248 (475 s)
epoch 6: loss 1.221 (570 s)
epoch 7: loss 1.201 (665 s)
epoch 8: loss 1.187 (759 s)
epoch 9: loss 1.178 (854 s)
epoch 10: loss 1.169 (949 s)
clean 79.2  linf 42.5  l2 62.7  l1 4

{'clean': 79.2, 'linf': 42.5, 'l2': 62.7, 'l1': 46.4, 'union': 40.6}

In [15]:
avg_mine = {key: (results_mine1[key] + results_mine2[key]) / 2 for key in results_mine1}

print(avg_mine)

{'clean': 76.65, 'linf': 43.95, 'l2': 61.7, 'l1': 46.7, 'union': 41.95}


In [13]:
model_ablation = adv_train(load_base_model(), my_loss, alpha=1.0)
test_model(model_ablation)


epoch 1: loss 1.368 (90 s)
epoch 2: loss 1.288 (179 s)
epoch 3: loss 1.260 (267 s)
epoch 4: loss 1.235 (356 s)
epoch 5: loss 1.206 (445 s)
epoch 6: loss 1.178 (534 s)
epoch 7: loss 1.148 (623 s)
epoch 8: loss 1.134 (712 s)
epoch 9: loss 1.119 (801 s)
epoch 10: loss 1.112 (889 s)
clean 75.9  linf 43.6  l2 61.6  l1 45.1  union 40.8   (135 s)


{'clean': 75.9, 'linf': 43.6, 'l2': 61.6, 'l1': 45.1, 'union': 40.8}

In [14]:
test_model(model_mine2, steps=200, restarts=5)

clean 77.0  linf 44.1  l2 61.8  l1 44.7  union 41.3   (1339 s)


{'clean': 77.0, 'linf': 44.1, 'l2': 61.8, 'l1': 44.7, 'union': 41.3}